In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#!/usr/bin/env python3
"""
Multi-Class Classification Pipeline - Task 2
"""

import os
import json
import logging
import random
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
import torchvision.transforms as transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, balanced_accuracy_score
)
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


@dataclass
class Config:
    """Configuration for multi-class training."""
    # Dataset paths
    nct_dataset_dir: str = "/kaggle/input/datasets/imrankhan77/nct-crc-he-100k/NCT-CRC-HE-100K"
    crc_dataset_dir: str = "/kaggle/input/datasets/imrankhan77/crc-val-he-7k/CRC-VAL-HE-7K"
    
    output_dir: str = "/kaggle/working/multiclass_outputs"
    
    # Training settings
    img_size: int = 224
    batch_size: int = 64
    num_workers: int = 4
    epochs: int = 10
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    
    # Split ratios
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    
    # Device
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    use_amp: bool = True
    seed: int = 42
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'models'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'figures'), exist_ok=True)


def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# DATA TRANSFORMS
# =============================================================================

def get_train_transforms(img_size: int = 224):
    """Training augmentation."""
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(90),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def get_val_transforms(img_size: int = 224):
    """Validation/test transforms."""
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


# =============================================================================
# DATASET WITH BINARY MAPPING
# =============================================================================

class MultiClassDataset(Dataset):
    """Dataset for multi-class classification."""
    def __init__(self, root_dir: str, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.labels = []
        self.class_names = []
        self._load_dataset()
    
    def _load_dataset(self):
        logger.info(f"Loading dataset from {self.root_dir}")
        class_dirs = sorted([d for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_names = [d.name for d in class_dirs]
        
        for class_idx, class_dir in enumerate(class_dirs):
            image_files = list(class_dir.glob('*'))
            for img_path in image_files:
                if img_path.suffix.lower() in ['.tif', '.png', '.jpg', '.jpeg']:
                    self.samples.append(str(img_path))
                    self.labels.append(class_idx)
        
        logger.info(f"  Loaded {len(self.samples)} images from {len(self.class_names)} classes")
        logger.info(f"  Classes: {self.class_names}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            logger.error(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224))
        
        if self.transform:
            image = self.transform(image)
        
        return image, label


# =============================================================================
# MODELS
# =============================================================================

class MultiClassClassifier(nn.Module):
    """Multi-class classifier for colon cancer images."""
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        weights = EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        self.backbone = efficientnet_v2_s(weights=weights)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        if features.dim() > 2:
            features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(features)


# =============================================================================
# TRAINER
# =============================================================================

class MultiClassTrainer:
    """Training engine for multi-class classification."""
    def __init__(self, model: nn.Module, train_loader: DataLoader, 
                 val_loader: DataLoader, config: Config):
        self.model = model.to(config.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = config.device
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(
            model.parameters(), 
            lr=config.learning_rate, 
            weight_decay=config.weight_decay
        )
        self.scheduler = OneCycleLR(
            self.optimizer, 
            max_lr=config.learning_rate,
            epochs=config.epochs,
            steps_per_epoch=len(train_loader)
        )
        self.scaler = GradScaler(enabled=config.use_amp)
        
        self.history = {
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': []
        }
        self.best_val_acc = 0.0
    
    def train_epoch(self, epoch: int):
        """Train one epoch."""
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs} [Train]")
        for images, labels in pbar:
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            self.optimizer.zero_grad()
            
            with autocast(enabled=self.config.use_amp):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
            
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()
            
            running_loss += loss.item() * images.size(0)
            
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
        
        epoch_loss = running_loss / len(self.train_loader.dataset)
        epoch_acc = correct / total
        return epoch_loss, epoch_acc
    
    @torch.no_grad()
    def validate(self):
        """Validate model."""
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in tqdm(self.val_loader, desc="[Validation]"):
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            with autocast(enabled=self.config.use_amp):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        val_loss = running_loss / len(self.val_loader.dataset)
        val_acc = correct / total
        return val_loss, val_acc
    
    def train(self):
        """Full training loop."""
        logger.info("\n" + "=" * 80)
        logger.info("STARTING MULTI-CLASS TRAINING")
        logger.info("=" * 80)
        
        for epoch in range(self.config.epochs):
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate()
            
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            
            logger.info(
                f"Epoch {epoch+1}/{self.config.epochs} | "
                f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%"
            )
            
            # Save best model
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.save_checkpoint(epoch)
                logger.info(f"  ✓ Best model saved (Val Acc: {val_acc*100:.2f}%)")
        
        logger.info(f"\n✓ Training complete! Best Val Acc: {self.best_val_acc*100:.2f}%")
        return self.history
    
    def save_checkpoint(self, epoch: int):
        """Save model checkpoint."""
        path = os.path.join(self.config.output_dir, 'models', 'best_multiclass_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_acc': self.best_val_acc,
        }, path)
    
    def load_best_model(self):
        """Load best checkpoint."""
        path = os.path.join(self.config.output_dir, 'models', 'best_multiclass_model.pth')
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        logger.info(f"✓ Loaded best model (Val Acc: {checkpoint['best_val_acc']*100:.2f}%)")


# =============================================================================
# EVALUATOR
# =============================================================================

class MultiClassEvaluator:
    """Evaluates multi-class classification."""
    def __init__(self, model: nn.Module, config: Config):
        self.model = model
        self.config = config
        self.device = config.device
        self.results = {}
    
    @torch.no_grad()
    def evaluate_dataset(self, dataloader: DataLoader, dataset_name: str, 
                        class_names: List[str]) -> Dict:
        """Evaluate on a single dataset."""
        logger.info(f"\n{'=' * 80}")
        logger.info(f"TESTING: {dataset_name}")
        logger.info(f"{'=' * 80}")
        
        self.model.eval()
        all_preds = []
        all_labels = []
        
        for images, labels in tqdm(dataloader, desc=f"Processing {dataset_name}"):
            images = images.to(self.device)
            outputs = self.model(images)
            preds = outputs.argmax(1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
        
        preds_np = np.array(all_preds)
        labels_np = np.array(all_labels)
        
        # Calculate metrics
        accuracy = accuracy_score(labels_np, preds_np)
        balanced_acc = balanced_accuracy_score(labels_np, preds_np)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_np, preds_np, average='weighted', zero_division=0
        )
        
        results = {
            'dataset': dataset_name,
            'num_samples': len(labels_np),
            'num_classes': len(class_names),
            'class_names': class_names,
            'accuracy': accuracy,
            'balanced_accuracy': balanced_acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'confusion_matrix': confusion_matrix(labels_np, preds_np).tolist(),
        }
        
        logger.info(f"\n📊 RESULTS:")
        logger.info(f"  Samples:           {len(labels_np)}")
        logger.info(f"  Classes:           {len(class_names)}")
        logger.info(f"  Accuracy:          {accuracy*100:.2f}%")
        logger.info(f"  Balanced Accuracy: {balanced_acc*100:.2f}%")
        logger.info(f"  Precision:         {precision:.4f}")
        logger.info(f"  Recall:            {recall:.4f}")
        logger.info(f"  F1-Score:          {f1:.4f}")
        
        self.results[dataset_name] = results
        return results
    
    def plot_comparison(self):
        """Visualize performance across test sets."""
        if not self.results:
            return
        
        datasets = list(self.results.keys())
        accuracies = [self.results[d]['accuracy'] * 100 for d in datasets]
        f1_scores = [self.results[d]['f1_score'] * 100 for d in datasets]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Accuracy comparison
        bars1 = ax1.bar(range(len(datasets)), accuracies, color='steelblue', edgecolor='navy', alpha=0.8)
        ax1.set_title('Test Set Accuracy Comparison', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Accuracy (%)', fontsize=12)
        ax1.set_xticks(range(len(datasets)))
        ax1.set_xticklabels(datasets, rotation=15, ha='right')
        ax1.set_ylim([0, 100])
        ax1.grid(axis='y', alpha=0.3)
        
        for i, v in enumerate(accuracies):
            ax1.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
        
        # F1-Score comparison
        bars2 = ax2.bar(range(len(datasets)), f1_scores, color='coral', edgecolor='darkred', alpha=0.8)
        ax2.set_title('Test Set F1-Score Comparison', fontsize=14, fontweight='bold')
        ax2.set_ylabel('F1-Score (%)', fontsize=12)
        ax2.set_xticks(range(len(datasets)))
        ax2.set_xticklabels(datasets, rotation=15, ha='right')
        ax2.set_ylim([0, 100])
        ax2.grid(axis='y', alpha=0.3)
        
        for i, v in enumerate(f1_scores):
            ax2.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
        
        plt.tight_layout()
        output_path = os.path.join(self.config.output_dir, 'figures', 'test_comparison.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"✓ Comparison plot saved: {output_path}")
        plt.close()
    
    def save_results(self):
        """Save results to JSON."""
        output_path = os.path.join(self.config.output_dir, 'multiclass_results.json')
        with open(output_path, 'w') as f:
            json.dump(self.results, f, indent=2)
        logger.info(f"✓ Results saved: {output_path}")


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    """Main execution."""
    config = Config()
    set_seed(config.seed)
    
    logger.info("\n" + "=" * 80)
    logger.info("MULTI-CLASS CLASSIFICATION - TASK 2")
    logger.info("=" * 80)
    logger.info(f"Device: {config.device}")
    
    # ========== STEP 1: LOAD AND SPLIT NCT DATASET ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 1: LOADING NCT-CRC-HE-100K DATASET")
    logger.info("=" * 80)
    
    train_transform = get_train_transforms(config.img_size)
    val_transform = get_val_transforms(config.img_size)
    
    # Load NCT dataset
    nct_dataset = MultiClassDataset(config.nct_dataset_dir, val_transform)
    num_classes = len(nct_dataset.class_names)
    logger.info(f"✓ NCT Dataset: {len(nct_dataset)} images, {num_classes} classes")
    logger.info(f"✓ Classes: {nct_dataset.class_names}")
    
    # Split NCT: 70% train, 15% val, 15% test
    nct_indices = np.arange(len(nct_dataset))
    nct_train_idx, nct_temp_idx = train_test_split(
        nct_indices,
        test_size=(config.val_ratio + config.test_ratio),
        stratify=nct_dataset.labels,
        random_state=config.seed
    )
    nct_val_idx, nct_test_idx = train_test_split(
        nct_temp_idx,
        test_size=config.test_ratio / (config.val_ratio + config.test_ratio),
        stratify=[nct_dataset.labels[i] for i in nct_temp_idx],
        random_state=config.seed
    )
    
    logger.info(f"✓ NCT Train: {len(nct_train_idx)} samples")
    logger.info(f"✓ NCT Val: {len(nct_val_idx)} samples")
    logger.info(f"✓ NCT Test: {len(nct_test_idx)} samples")
    
    # ========== STEP 2: LOAD AND SPLIT CRC DATASET ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 2: LOADING CRC-VAL-HE-7K DATASET")
    logger.info("=" * 80)
    
    crc_dataset = MultiClassDataset(config.crc_dataset_dir, val_transform)
    logger.info(f"✓ CRC Dataset: {len(crc_dataset)} images, {len(crc_dataset.class_names)} classes")
    logger.info(f"✓ Classes: {crc_dataset.class_names}")
    
    # Split CRC: 70% train, 15% val, 15% test
    crc_indices = np.arange(len(crc_dataset))
    crc_train_idx, crc_temp_idx = train_test_split(
        crc_indices,
        test_size=(config.val_ratio + config.test_ratio),
        stratify=crc_dataset.labels,
        random_state=config.seed
    )
    crc_val_idx, crc_test_idx = train_test_split(
        crc_temp_idx,
        test_size=config.test_ratio / (config.val_ratio + config.test_ratio),
        stratify=[crc_dataset.labels[i] for i in crc_temp_idx],
        random_state=config.seed
    )
    
    logger.info(f"✓ CRC Train: {len(crc_train_idx)} samples")
    logger.info(f"✓ CRC Val: {len(crc_val_idx)} samples")
    logger.info(f"✓ CRC Test: {len(crc_test_idx)} samples")
    
    # ========== STEP 3: CREATE COMBINED TRAINING SET ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 3: CREATING COMBINED TRAINING SET (NCT + CRC)")
    logger.info("=" * 80)
    
    # Create training datasets with augmentation
    nct_train_dataset = MultiClassDataset(config.nct_dataset_dir, train_transform)
    crc_train_dataset = MultiClassDataset(config.crc_dataset_dir, train_transform)
    
    # Create validation datasets (no augmentation)
    nct_val_dataset = MultiClassDataset(config.nct_dataset_dir, val_transform)
    crc_val_dataset = MultiClassDataset(config.crc_dataset_dir, val_transform)
    
    # Combine train and val subsets
    train_subset = torch.utils.data.ConcatDataset([
        torch.utils.data.Subset(nct_train_dataset, nct_train_idx),
        torch.utils.data.Subset(crc_train_dataset, crc_train_idx)
    ])
    
    val_subset = torch.utils.data.ConcatDataset([
        torch.utils.data.Subset(nct_val_dataset, nct_val_idx),
        torch.utils.data.Subset(crc_val_dataset, crc_val_idx)
    ])
    
    train_loader = DataLoader(
        train_subset, batch_size=config.batch_size, shuffle=True,
        num_workers=config.num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_subset, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, pin_memory=True
    )
    
    logger.info(f"✓ Combined Training samples: {len(train_subset)}")
    logger.info(f"✓ Combined Validation samples: {len(val_subset)}")
    
    # ========== STEP 4: TRAIN MODEL ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 4: TRAINING MODEL ON COMBINED DATASET")
    logger.info("=" * 80)
    
    model = MultiClassClassifier(num_classes=num_classes, pretrained=True)
    trainer = MultiClassTrainer(model, train_loader, val_loader, config)
    history = trainer.train()
    trainer.load_best_model()
    
    # ========== STEP 5: PREPARE TEST SETS ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 5: PREPARING TEST SETS")
    logger.info("=" * 80)
    
    # NCT test set
    nct_test_subset = torch.utils.data.Subset(nct_val_dataset, nct_test_idx)
    nct_test_loader = DataLoader(
        nct_test_subset, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, pin_memory=True
    )
    
    # CRC test set
    crc_test_subset = torch.utils.data.Subset(crc_val_dataset, crc_test_idx)
    crc_test_loader = DataLoader(
        crc_test_subset, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, pin_memory=True
    )
    
    # Combined test set (NCT + CRC)
    combined_test_subset = torch.utils.data.ConcatDataset([nct_test_subset, crc_test_subset])
    combined_test_loader = DataLoader(
        combined_test_subset, batch_size=config.batch_size, shuffle=False,
        num_workers=config.num_workers, pin_memory=True
    )
    
    logger.info(f"✓ NCT Test Set: {len(nct_test_subset)} samples")
    logger.info(f"✓ CRC Test Set: {len(crc_test_subset)} samples")
    logger.info(f"✓ Combined Test Set: {len(combined_test_subset)} samples")
    
    # ========== STEP 6: EVALUATE ON ALL TEST SETS ==========
    logger.info("\n" + "=" * 80)
    logger.info("STEP 6: EVALUATING ON TEST SETS")
    logger.info("=" * 80)
    
    evaluator = MultiClassEvaluator(trainer.model, config)
    
    # Test on NCT
    evaluator.evaluate_dataset(nct_test_loader, 'NCT-CRC-HE-100K (Test)', nct_dataset.class_names)
    
    # Test on CRC
    evaluator.evaluate_dataset(crc_test_loader, 'CRC-VAL-HE-7K (Test)', crc_dataset.class_names)
    
    # Test on Combined
    evaluator.evaluate_dataset(combined_test_loader, 'Combined (NCT + CRC Test)', nct_dataset.class_names)
    
    # Save results
    evaluator.save_results()
    evaluator.plot_comparison()
    
    logger.info("\n" + "=" * 80)
    logger.info("✓ MULTI-CLASS EXPERIMENT COMPLETE!")
    logger.info("=" * 80)


if __name__ == "__main__":
    main()


2026-04-03 16:45:48,489 - INFO - 
2026-04-03 16:45:48,490 - INFO - MULTI-CLASS CLASSIFICATION - TASK 2
2026-04-03 16:45:48,491 - INFO - ================================================================================
2026-04-03 16:45:48,491 - INFO - Device: cuda
2026-04-03 16:45:48,492 - INFO - 
2026-04-03 16:45:48,492 - INFO - STEP 1: LOADING NCT-CRC-HE-100K DATASET
2026-04-03 16:45:48,493 - INFO - ================================================================================
2026-04-03 16:45:48,494 - INFO - Loading dataset from /kaggle/input/datasets/imrankhan77/nct-crc-he-100k/NCT-CRC-HE-100K
2026-04-03 16:45:50,246 - INFO -   Loaded 100000 images from 9 classes
2026-04-03 16:45:50,247 - INFO -   Classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
2026-04-03 16:45:50,249 - INFO - ✓ NCT Dataset: 100000 images, 9 classes
2026-04-03 16:45:50,250 - INFO - ✓ Classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
2026-04-03 16:45:50,310 - INFO 

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 196MB/s] 
/tmp/ipykernel_55/1070984080.py:212: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=config.use_amp)
2026-04-03 16:45:53,452 - INFO - 
2026-04-03 16:45:53,453 - INFO - STARTING MULTI-CLASS TRAINING
2026-04-03 16:45:53,453 - INFO - ================================================================================
Epoch 1/10 [Train]:   0%|          | 0/1173 [00:00<?, ?it/s]/tmp/ipykernel_55/1070984080.py:234: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.config.use_amp):
[Validation]:   0%|          | 0/252 [00:00<?, ?it/s]/tmp/ipykernel_55/1070984080.py:270: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.config.use_am

In [2]:
import shutil
import os
from IPython.display import FileLink

# 1. Path to your outputs
output_path = "/kaggle/working/multiclass_outputs"
zip_name = 'colon_cancer_multi_results'

# 2. Create the zip file
# This takes everything in /outputs and puts it into colon_cancer_results.zip
shutil.make_archive(zip_name, 'zip', output_path)

print(f"✅ Created {zip_name}.zip")

# 3. Generate the download link
FileLink(f'{zip_name}.zip')

✅ Created colon_cancer_multi_results.zip


/kaggle/working/colon_cancer_multi_results.zip